## 1. Load the house data

Read the CSV file. Each row is a house with 8 inputs and its price.
We will predict the price.

In [1]:
import pandas as pd
data = pd.read_csv("../house_data.csv")

In [2]:
data.head()

,sqft,bedrooms,bathrooms,age,floors,garage,lot_size,city_index,price
0,1360,2,4,18,1,0,5566,1,248990
1,4272,3,4,29,2,2,2909,0,941769
2,3592,1,4,13,2,1,7364,8,378015
3,966,1,4,19,1,0,3581,0,196887
4,4926,2,4,48,1,1,1691,3,1063441


In [3]:
type(data)

pandas.DataFrame

## 2. Separate the inputs (X) and the output (Y)

Y is the price. X is the 8 other columns.
Check the shapes: 1000 rows, 8 inputs, 1 output.

In [4]:
X = data.drop(columns=['price'])

In [5]:
Y = data[['price']]

In [6]:
X.shape

(1000, 8)

In [7]:
Y.shape

(1000, 1)

In [8]:
type(X)

pandas.DataFrame

## 3. Convert to NumPy float32

PyTorch works with float32 numbers, so convert the tables first.
to_numpy() turns a DataFrame into an array.

In [9]:
import numpy as np

In [10]:
X_new = X.to_numpy(dtype=np.float32)
Y_new = Y.to_numpy(dtype=np.float32)

In [11]:
X_new

array([[1.360e+03, 2.000e+00, 4.000e+00, ..., 0.000e+00, 5.566e+03,
        1.000e+00],
       [4.272e+03, 3.000e+00, 4.000e+00, ..., 2.000e+00, 2.909e+03,
        0.000e+00],
       [3.592e+03, 1.000e+00, 4.000e+00, ..., 1.000e+00, 7.364e+03,
        8.000e+00],
       ...,
       [2.606e+03, 4.000e+00, 4.000e+00, ..., 1.000e+00, 1.495e+03,
        0.000e+00],
       [4.723e+03, 5.000e+00, 2.000e+00, ..., 0.000e+00, 3.148e+03,
        9.000e+00],
       [3.268e+03, 4.000e+00, 3.000e+00, ..., 2.000e+00, 1.556e+03,
        2.000e+00]], shape=(1000, 8), dtype=float32)

In [12]:
X_new.dtype

dtype('float32')

## 4. Split into training and test data

Use 80% of the houses for training and keep 20% for testing.
random_state=42 makes the split repeatable.

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
x_train, x_test, y_train, y_test = train_test_split(X_new, Y_new, test_size=0.2, random_state=42)

## 5. Scale the numbers (StandardScaler)

Features such as sqft and bedrooms are on very different scales, which makes training slow and unstable.
Fit the scaler on the training data only, then transform both parts.

In [15]:
from sklearn.preprocessing import StandardScaler

In [16]:
x_scalar = StandardScaler().fit(x_train)
y_scalar = StandardScaler().fit(y_train)

In [17]:
x_scalar

StandardScaler()

In [18]:
x_train_scalar = x_scalar.transform(x_train).astype(np.float32)
x_test_scalar = x_scalar.transform(x_test).astype(np.float32)
y_train_scalar = y_scalar.transform(y_train).astype(np.float32)
y_test_scalar = y_scalar.transform(y_test).astype(np.float32)

In [19]:
x_train_scalar

array([[-0.6277223 , -0.6819861 , -0.44865718, ..., -0.04929634,
        -0.06389936, -1.5591687 ],
       [ 0.56318504,  1.4299709 , -1.3240858 , ...,  1.1831132 ,
         0.6206979 ,  0.48285297],
       [ 1.3523309 , -0.6819861 , -0.44865718, ...,  1.1831132 ,
        -1.475833  ,  1.1635269 ],
       ...,
       [-1.0622318 ,  1.4299709 ,  0.4267714 , ..., -0.04929634,
        -1.2116971 , -1.2188318 ],
       [-0.7579154 ,  0.7259852 ,  0.4267714 , ..., -1.281706  ,
         0.7054062 , -1.2188318 ],
       [-1.1588782 ,  0.7259852 , -1.3240858 , ..., -1.281706  ,
        -1.170113  ,  0.8231899 ]], shape=(800, 8), dtype=float32)

## 6. Install and import PyTorch

PyTorch is the library for building neural networks.
Install it once with pip.

In [20]:
!python3 -m pip install torch



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [21]:
import torch

## 7. Convert the arrays to tensors

A tensor is the PyTorch version of an array.
The model only accepts tensors.

In [22]:
x_train_tensor = torch.from_numpy(x_train_scalar)
x_test_tensor = torch.from_numpy(x_test_scalar)
y_train_tensor = torch.from_numpy(y_train_scalar)
y_test_tensor= torch.from_numpy(y_test_scalar)

In [23]:
x_train_tensor

tensor([[-0.6277, -0.6820, -0.4487,  ..., -0.0493, -0.0639, -1.5592],
        [ 0.5632,  1.4300, -1.3241,  ...,  1.1831,  0.6207,  0.4829],
        [ 1.3523, -0.6820, -0.4487,  ...,  1.1831, -1.4758,  1.1635],
        ...,
        [-1.0622,  1.4300,  0.4268,  ..., -0.0493, -1.2117, -1.2188],
        [-0.7579,  0.7260,  0.4268,  ..., -1.2817,  0.7054, -1.2188],
        [-1.1589,  0.7260, -1.3241,  ..., -1.2817, -1.1701,  0.8232]])

In [24]:
x_train_tensor.shape

torch.Size([800, 8])

## 8. Define the neural network (with hidden layers)

The data passes through hidden layers (16, 8 and 4 neurons) with ReLU between them.
Hidden layers let the model learn curved, non-linear patterns.

In [25]:
import torch.nn as nn
import torch.optim as optim

In [26]:
# Option 2: network with hidden layers
torch.manual_seed(42)

class HousePricePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(8,16),
            nn.ReLU(),
            nn.Linear(16,8),
            nn.ReLU(),
            nn.Linear(8,4),
            nn.ReLU(),
            nn.Linear(4,1)
        )
    def forward(self, x_train_data):
        return self.network(x_train_data)

model = HousePricePredictor()


In [28]:
model

HousePricePredictor(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=4, bias=True)
    (5): ReLU()
    (6): Linear(in_features=4, out_features=1, bias=True)
  )
)

In [29]:
for param in model.parameters():
    print (param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076],
        [ 0.3117, -0.2594,  0.3073,  0.0662,  0.2612,  0.0479,  0.1705, -0.0499],
        [ 0.2725,  0.0523, -0.1651,  0.0901, -0.1629, -0.0415, -0.1436,  0.2345],
        [-0.2791, -0.1630, -0.0998, -0.2126,  0.0334, -0.3492,  0.3193, -0.3003],
        [ 0.2730,  0.0588, -0.1148,  0.2185,  0.0551,  0.2857,  0.0387, -0.1115],
        [ 0.0950, -0.0959,  0.1488,  0.3157,  0.2044, -0.1546,  0.2041,  0.0633],
        [ 0.1795, -0.2155, -0.3500, -0.1366, -0.2712,  0.2901,  0.1018,  0.1464],
        [ 0.1118, -0.0062,  0.2767, -0.2512,  0.0223, -0.2413,  0.1090, -0.1218],
        [ 0.1083, -0.0737,  0.2932, -0.2096, -0.2109, -0.2109,  0.3180,  0.1178],
        [ 0.3402, -0.2918, -0.3507, -0.2766, -0.2378,  0.1432,  0.1266,  0.2938],
        [-0.1826, -0.2410,  0.1876, -0.1429,  0.2146, -0.0839,  0.2022, -0.2747],
        [-0.1784,  0.1078,  0.0747, -0.0901,  0.2107,  0.2403, -0.2564, -0.1

## 9. Choose the loss function

MSELoss measures how far the predictions are from the real prices.
Training tries to make this number small.

In [30]:
criterion = nn.MSELoss()

## 10. Choose the optimizer

Adam updates the weights to reduce the loss.
lr=0.001 is the learning rate, the size of each update.

In [31]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 11. Train the model

Each epoch does a forward pass, calculates the loss, goes backward and updates the weights.
The loss should go down as it learns.

In [32]:
model.train()

HousePricePredictor(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=4, bias=True)
    (5): ReLU()
    (6): Linear(in_features=4, out_features=1, bias=True)
  )
)

In [33]:
epochs = 1000

In [35]:
for epoch in range(1, epochs+1):
    #1. Forward Pass
    y_predictions = model(x_train_tensor)
    #2. Calculate loss
    loss = criterion(y_predictions, y_train_tensor )
    #3. Optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{epochs} | loss: {loss.item():.6f}")
    

Epoch  100/1000 | loss: 0.155989
Epoch  200/1000 | loss: 0.150415
Epoch  300/1000 | loss: 0.145797
Epoch  400/1000 | loss: 0.140299
Epoch  500/1000 | loss: 0.136023
Epoch  600/1000 | loss: 0.131829
Epoch  700/1000 | loss: 0.127513
Epoch  800/1000 | loss: 0.121898
Epoch  900/1000 | loss: 0.117585
Epoch 1000/1000 | loss: 0.114372


## 12. Look at the learned weights

Every layer now has its own weights and biases.
There are many more numbers than in the simple model.

In [36]:
for param in model.parameters():
    print (param)

Parameter containing:
tensor([[ 0.2754,  0.4406,  0.1662,  0.1995, -0.0550, -0.1275, -0.5817, -0.2066],
        [ 0.2727, -0.9562,  0.8627,  0.4472,  0.9723, -0.4174, -0.1392,  0.9830],
        [ 0.3322,  0.8729,  0.7828,  0.1938,  0.4360, -0.5206, -0.7619,  0.5788],
        [-0.4622,  0.2000,  0.0267, -0.2017, -0.1340, -0.3255,  0.0322, -0.2802],
        [ 0.5293,  0.1514, -0.2304,  0.3113,  0.5216,  0.5469,  0.0224,  0.0057],
        [ 0.3749, -0.3227,  0.0140, -0.1460,  0.1440, -0.0663,  0.6336,  0.0366],
        [ 0.8844, -0.2553, -0.4918, -0.2102, -0.3836,  0.6843, -0.2690, -0.2918],
        [ 0.3237,  0.2673,  0.9196, -0.4339, -0.2429, -0.6739,  0.1821,  0.0281],
        [ 0.2475,  0.2315,  0.6305, -0.3771, -0.3179, -0.8335,  0.4798,  0.5203],
        [ 0.5563, -0.0385, -0.7034, -0.5580, -0.4557,  0.4474, -0.2577,  0.1799],
        [-0.3578, -0.4856,  0.0251,  0.0338,  0.3905,  0.0659, -0.2501,  0.0722],
        [-0.3755,  0.0763, -0.1068,  0.1735, -0.1152,  0.1315, -0.1235,  0.0

## 13. Predict on the test data

eval() and no_grad() switch off training behaviour.
The predictions are still scaled, so convert them back to dollars next.

In [37]:
model.eval()
with torch.no_grad():
    y_pred_test = model(x_test_tensor)
    y_pre_numpy = y_pred_test.numpy()
y_pre_numpy

array([[ 0.6100136 ],
       [-0.27614   ],
       [ 1.4250414 ],
       [ 1.5729532 ],
       [ 1.2618124 ],
       [ 0.48152635],
       [ 1.1112165 ],
       [ 1.3432227 ],
       [ 0.7907834 ],
       [ 1.3903453 ],
       [ 0.4729803 ],
       [ 0.32815215],
       [-1.4073089 ],
       [ 0.89627624],
       [-0.26336005],
       [-1.3067622 ],
       [-0.71613395],
       [-1.1481746 ],
       [ 0.46450213],
       [-0.75022984],
       [ 1.1244464 ],
       [-0.45354185],
       [ 0.24188414],
       [ 0.1409364 ],
       [-0.18592516],
       [-1.1007867 ],
       [-0.43503806],
       [-0.5458247 ],
       [-1.1910284 ],
       [-0.59998983],
       [-1.0962058 ],
       [ 0.6522213 ],
       [ 0.21306208],
       [-1.0363615 ],
       [-0.52921677],
       [ 0.9078721 ],
       [-0.62372005],
       [ 0.3079721 ],
       [-0.83016133],
       [-1.3583337 ],
       [-1.2255433 ],
       [-1.2073755 ],
       [-0.41969264],
       [-0.85811436],
       [ 1.8798504 ],
       [ 1

## 14. Convert predictions back to dollars

inverse_transform undoes the scaling we applied to the prices.
Now the predictions are real prices.

In [38]:
y_pred_test = y_scalar.inverse_transform(y_pre_numpy).flatten()
y_pred_test

array([581855.44, 393951.03, 754678.  , 786041.94, 720066.1 , 554610.4 ,
       688133.  , 737328.75, 620186.75, 747320.9 , 552798.25, 522088.12,
       154092.34, 642556.  , 396660.94, 175412.78, 300652.53, 209040.47,
       551000.44, 293422.66, 690938.3 , 356333.88, 503795.44, 482389.97,
       413080.62, 219088.84, 360257.5 , 336765.75, 199953.55, 325280.3 ,
       220060.19, 590805.4 , 497683.88, 232749.89, 340287.38, 645014.8 ,
       320248.44, 517809.06, 276473.6 , 164477.28, 192634.84, 196487.22,
       363511.4 , 270546.3 , 851117.94, 842291.9 , 526850.5 , 517272.06,
       845306.9 , 422510.62, 164574.06, 441780.78, 446965.75, 507779.56,
       388394.4 , 667828.9 , 122057.97, 161698.34, 571324.75, 121462.72,
       685444.4 , 217477.64, 172173.84, 426035.4 , 487137.9 , 459230.5 ,
       338664.12, 462538.9 , 692133.06, 505354.56, 340254.28, 737217.2 ,
       267168.3 , 397753.  , 510238.28, 518544.12, 395749.47, 132355.  ,
       577242.1 , 761435.9 , 408216.34, 239150.89, 

## 15. Evaluate

R2 and the average error show how good the predictions are.
Compare the scores with the simple model.

In [39]:
from sklearn.metrics import r2_score, mean_absolute_error

y_actual = y_test.flatten()
print("R2 score :", round(r2_score(y_actual, y_pred_test), 3))
print("Avg error: $", round(mean_absolute_error(y_actual, y_pred_test)))

R2 score : 0.678
Avg error: $ 98510


In [40]:
pd.DataFrame({"actual": y_actual[:8].round(), "predicted": y_pred_test[:8].round()})

,actual,predicted
0,746264.0,581855.0
1,349011.0,393951.0
2,853367.0,754678.0
3,784465.0,786042.0
4,342987.0,720066.0
5,563293.0,554610.0
6,688837.0,688133.0
7,849834.0,737329.0


## 16. Save the model and the scalers

Save the trained weights (.pt) and both scalers (.pkl).
A new input must be scaled the same way, and the output converted back to dollars.

In [41]:
import os, joblib

network = "hidden"      # "simple" if you ran Option 1, "hidden" if you ran Option 2

os.makedirs("model", exist_ok=True)
torch.save(model.state_dict(), f"model/house_model_{network}.pt")   # the trained weights
joblib.dump(x_scalar, "model/x_scaler.pkl")                          # needed to scale new inputs
joblib.dump(y_scalar, "model/y_scaler.pkl")                          # needed to turn the output back into dollars
os.listdir("model")

['x_scaler.pkl', 'house_model_hidden.pt', 'y_scaler.pkl']